# Machine Learning 2 - HW 2

## Submission Instructions

This notebook contains all the questions. You should follow the instructions below.
Solutions for both theoretical and practical parts should be written in this notebook.

You should submit three files:
- IPYNB notebook:
  - All the wet and dry parts, including code, graphs, discussion, etc.
- Model weights:
  - Export the model's weights after training using a `.pt` or `.pkl` file (question 2).
- PDF file:
  - Export the notebook to PDF. Make sure that all the cells are visible.

All files should be in the following format: "HW2_ID1_ID2.file"

Good Luck!

# Question 1 - Generalization and Overfit (20 pt)

In this exercise, we will demonstrate overfitting to random labels. The settings are the following:

<li> Use the MNIST dataset.
<li> For model training - work on the first 128 samples from the train split.
<li> Fix the following parameters:
    <ul>
        <li> Shuffle to False.
        <li> Batch size to 128.
        <li> Use seed = 42 for reproducibility.
    </ul>

Generate random labels from Bernoulli distribution with a probability of $\frac{1}{2}$ (I.e., each sample is assigned a random label which is zero or one).

Show that by using a simple Fully Connected netwrok and cross-entropy loss, you are able to achieve a loss value of $\approx 0$ (the lower the better).

Plot the accuracy and loss convergence for this data and the test split of our dataset as a function of epochs.

Answer the following questions:

1. Describe the accuracy value of the test data as a function of epochs and explain the phenomenon.

2. Discuss the connection between model size (complexity) and the dataset size. How does it relate to generalization and overfitting?

3. Discuss general methods to avoid overfitting. Specially, discuss methods in three different levels: data, model, and training.

4. Find one academic paper that presents or analyzes a method for reducing overfitting in deep learning.

    The method may be: general or domain-specific, architecture-specific (e.g., CNNs, Transformers), or applied to a particular task (e.g., vision, NLP, biomedical data).

    Briefly summarize:

    - The overfitting problem addressed in the paper (if there is one)

    - The proposed method for reducing overfitting

    - The main findings or conclusions

    Include a link for the paper.

    For example, you might delve into the connection between model size (complexity) and the dataset size stated in https://arxiv.org/pdf/2001.08361
</p>

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms
import matplotlib.pyplot as plt
import numpy as np

# Set seed for reproducibility
torch.manual_seed(42)
np.random.seed(42)

# Check for GPU
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# Define transforms
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,))
])

# Load MNIST dataset
train_dataset_full = datasets.MNIST(root='./data', train=True, download=True, transform=transform)
test_dataset = datasets.MNIST(root='./data', train=False, download=True, transform=transform)

# Take only first 128 samples for training
train_dataset = Subset(train_dataset_full, range(128))

# Generate random labels from Bernoulli distribution with p=0.5
torch.manual_seed(42)
random_train_labels = torch.bernoulli(torch.full((128,), 0.5)).long()
random_test_labels = torch.bernoulli(torch.full((len(test_dataset),), 0.5)).long()

print(f"Training samples: {len(train_dataset)}")
print(f"Test samples: {len(test_dataset)}")
print(f"Train random labels: {random_train_labels.sum().item()} ones, {128 - random_train_labels.sum().item()} zeros")

# Custom Dataset with random labels
class RandomLabelDataset(torch.utils.data.Dataset):
    def __init__(self, original_dataset, random_labels):
        self.original_dataset = original_dataset
        self.random_labels = random_labels

    def __len__(self):
        return len(self.original_dataset)

    def __getitem__(self, idx):
        image, _ = self.original_dataset[idx]
        return image, self.random_labels[idx]

# Create datasets with random labels
train_dataset_random = RandomLabelDataset(train_dataset, random_train_labels)
test_dataset_random = RandomLabelDataset(test_dataset, random_test_labels)

# Create DataLoaders (shuffle=False as required)
train_loader = DataLoader(train_dataset_random, batch_size=128, shuffle=False)
test_loader = DataLoader(test_dataset_random, batch_size=128, shuffle=False)

# Model
class FullyConnectedNet(nn.Module):
    def __init__(self, input_size=784, hidden_sizes=[512, 256, 128], num_classes=2):
        super(FullyConnectedNet, self).__init__()
        layers = []
        prev_size = input_size
        for hidden_size in hidden_sizes:
            layers.append(nn.Linear(prev_size, hidden_size))
            layers.append(nn.ReLU())
            prev_size = hidden_size
        layers.append(nn.Linear(prev_size, num_classes))
        self.network = nn.Sequential(*layers)

    def forward(self, x):
        x = x.view(x.size(0), -1)
        return self.network(x)

model = FullyConnectedNet().to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

print(model)
print(f"\nTotal parameters: {sum(p.numel() for p in model.parameters()):,}")

# Training function
def train_epoch(model, loader, criterion, optimizer, device):
    model.train()
    total_loss = 0
    correct = 0
    total = 0

    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        _, predicted = outputs.max(1)
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()

    return total_loss / len(loader), 100. * correct / total

# Evaluation function
def evaluate(model, loader, criterion, device):
    model.eval()
    total_loss = 0
    correct = 0
    total = 0

    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)

            outputs = model(images)
            loss = criterion(outputs, labels)

            total_loss += loss.item()
            _, predicted = outputs.max(1)
            total += labels.size(0)
            correct += predicted.eq(labels).sum().item()

    return total_loss / len(loader), 100. * correct / total

# Training loop
num_epochs = 75
train_losses, train_accuracies = [], []
test_losses, test_accuracies = [], []

print("\nTraining on random labels...")
print("-" * 70)

for epoch in range(num_epochs):
    train_loss, train_acc = train_epoch(model, train_loader, criterion, optimizer, device)
    test_loss, test_acc = evaluate(model, test_loader, criterion, device)

    train_losses.append(train_loss)
    train_accuracies.append(train_acc)
    test_losses.append(test_loss)
    test_accuracies.append(test_acc)

    print(f"Epoch [{epoch+1:2d}/{num_epochs}] | Train Loss: {train_loss:.6f} | Train Acc: {train_acc:.2f}% | Test Acc: {test_acc:.2f}%")

print("-" * 70)
print(f"Final Train Loss: {train_losses[-1]:.6f}")
print(f"Final Train Accuracy: {train_accuracies[-1]:.2f}%")
print(f"Final Test Accuracy: {test_accuracies[-1]:.2f}%")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Plot Loss
axes[0].plot(train_losses, label='Train Loss (Random Labels)', color='blue', linewidth=2)
axes[0].plot(test_losses, label='Test Loss (Random Labels)', color='red', linewidth=2, alpha=0.7)
axes[0].set_xlabel('Epoch', fontsize=12)
axes[0].set_ylabel('Loss', fontsize=12)
axes[0].set_title('Loss Convergence', fontsize=14)
axes[0].legend(fontsize=10)
axes[0].grid(True, alpha=0.3)
axes[0].set_yscale('log')

# Plot Accuracy
axes[1].plot(train_accuracies, label='Train Accuracy (Random Labels)', color='blue', linewidth=2)
axes[1].plot(test_accuracies, label='Test Accuracy (Random Labels)', color='red', linewidth=2, alpha=0.7)
axes[1].axhline(y=50, color='green', linestyle='--', label='Random Chance (50%)', alpha=0.7)
axes[1].set_xlabel('Epoch', fontsize=12)
axes[1].set_ylabel('Accuracy (%)', fontsize=12)
axes[1].set_title('Accuracy Convergence', fontsize=14)
axes[1].legend(fontsize=10)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Answers

### 1. Describe the accuracy value of the test data as a function of epochs and explain the phenomenon.

The **test accuracy stays constant at ~50%** throughout all epochs, while **train accuracy reaches 100%** by epoch 20.

**Why this happens:**
- The model memorized all 128 training samples with their random labels
- Since labels are random, there's no real pattern connecting images to labels
- On new test images, the model just guesses randomly → 50% accuracy (coin flip)

**Conclusion:** The model overfit to meaningless data. Low training loss ≠ good generalization.

---

### 2. Discuss the connection between model size and dataset size. How does it relate to generalization and overfitting?

**Our experiment:** ~400,000 parameters vs 128 samples (ratio 3000:1)

**The connection:**
- **Too many parameters, too little data** → model memorizes instead of learning patterns → **overfitting**
- **Too few parameters** → model can't capture patterns → **underfitting**
- **Balanced ratio** → model learns generalizable patterns → **good generalization**

**Rule of thumb:** Need much more data than parameters for good generalization.

---

### 3. Discuss general methods to avoid overfitting at three levels.

**Data Level:**
- Data augmentation (rotate, flip, crop images)
- Collect more data
- Use pretrained models (transfer learning)

**Model Level:**
- Reduce model size (fewer layers/neurons)
- Dropout (randomly turn off neurons during training)
- Batch normalization

**Training Level:**
- Early stopping (stop when validation loss increases)
- L2 regularization / weight decay (penalize large weights)
- Learning rate scheduling
---
### 4. Academic Paper on Reducing Overfitting

**Paper:** "Dropout: A Simple Way to Prevent Neural Networks from Overfitting"

**Authors:** Srivastava, Hinton, Krizhevsky, Sutskever, Salakhutdinov (2014)

**Link:** jmlr.org/papers/volume15/srivastava14a/srivastava14a.pdf


**The overfitting problem addressed:**
- Deep neural networks with many parameters easily memorize training data
- Neurons develop complex co-adaptations that don't generalize to new data
- Small datasets make this problem worse

**The proposed method:**
- **Dropout** randomly sets a fraction of neurons to zero during each training step (typically 50%)
- Each training iteration uses a different "thinned" network
- At test time, all neurons are used but weights are scaled down

**Main findings:**
- Dropout significantly reduces overfitting across many tasks
- Achieved state-of-the-art results on MNIST, CIFAR, ImageNet, and speech recognition
- Works like training an ensemble of many networks efficiently
- Simple to implement with minimal computational cost

********************

# Question 2 - Sentiment Analysis - Classification (70 pt)

The goal of this exercise is to get familiar with recurrent neural networks.
<br><br>
Text emotion recognition, AKA sentiment analysis, is a rapidly growing field due to its practical applications. For instance, companies may be interested in classifying customer reviews as positive or negative to understand user satisfaction.
<br><br>
In this exercise, you will detect the emotion of a sentence. You will implement two models - RNN and LSTM. You need to reach at least 51% and 70% accuracy on the test set with RNN-based and LSTM-based models, respectively.
<br><br>
You should:
  <li> Use different optimization and regularization methods. </li>
  <li> Try different combinations of hyperparamters.</li>
</ul>
</p>

<br>

### Data

You will work with IMDB dataset, containing movie reviews and their labels according to the emotion conveyed in the review [positive, negative]. The dataset is divided into train and test sets (folders), each contains a folder for positive and negative reviews.
<br>

The dataset vocabulary is given (`imdb.vocab`). Feel free to use it or any other custom vocabulary.

### Tips


- Instead of using One-hot embeddings for the data, use <code>nn.Embedding</code>.

- Tokenization ("model language for text") - use word-based tokenization (each word is a token).

- You also might add "Start Of Sentence" (SOS) and "End Of Sentence" (EOS), "padding" (PAD) and "unknown" (UNK) tokens (self-learning). Note, to use them you have to insert them manually to your vocabulary and to the dataset's text.

    You are given `VocabEmbedding` for word-embedding management. The class loads the dataset's vocabulary and assigns an embedding for each word in it (+ special tokens). Feel free to use it as part of your model. Note, the embeddings are trainable and the class is a torch.nn class.

- Use GPU.

- It's recommended to use Deep RNN\LSTM (number of recurrent layers stacked on top of each other). For example, using `num_layers` in `nn.RNN`.

### a) EDA (10 pt)

Explore and analyze your data. Explain your data cleaning and processing pipeline.

## Imports

In [ ]:
import tarfile
import os
import random
import re
from collections import Counter
import numpy as np
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.optim as optim
import copy
from tqdm.notebook import tqdm
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import classification_report
import seaborn as sns

## Unzip data

In [ ]:
# Open the tar.gz file
with tarfile.open('/content/aclImdb_v1.tar.gz', 'r:gz') as tar:
    # Extract all contents to the current directory with filter='data' to address DeprecationWarning
    tar.extractall('/content/', filter='data')

## Load data

In [ ]:
def load_imdb_data(directory_path, label):
    reviews = []
    for filename in os.listdir(directory_path):
        if filename.endswith(".txt"):
            filepath = os.path.join(directory_path, filename)
            with open(filepath, "r", encoding="utf-8") as f:
                review_text = f.read()
                reviews.append((review_text, label))
    return reviews

random.seed(42)

# Load training data
train_pos_reviews = load_imdb_data('/content/aclImdb/train/pos', 1)
train_neg_reviews = load_imdb_data('/content/aclImdb/train/neg', 0)
train_data = train_pos_reviews + train_neg_reviews
random.shuffle(train_data)

# Load testing data
test_pos_reviews = load_imdb_data('/content/aclImdb/test/pos', 1)
test_neg_reviews = load_imdb_data('/content/aclImdb/test/neg', 0)
test_data = test_pos_reviews + test_neg_reviews
random.shuffle(test_data)

# Print counts
print(f"Number of training reviews: {len(train_data)}")
print(f"Number of testing reviews: {len(test_data)}")


## Tokenize and clean special tokens

In [ ]:
def tokenize_review_basic(review_text):
    # Convert to lowercase and split by non-alphanumeric characters
    tokens = re.findall(r'\b\w+\b', review_text.lower())

    # Define special tokens to be excluded
    special_tokens_to_exclude = {'<pad>', '<unk>', '<sos>', '<eos>', 'br', 's', 't'}

    # Filter out special tokens
    filtered_tokens = [token for token in tokens if token not in special_tokens_to_exclude]
    return filtered_tokens


# Apply tokenization to training data
tokenized_train_data_words = []
for review_text, label in train_data:
    tokenized_review = tokenize_review_basic(review_text)
    tokenized_train_data_words.append((tokenized_review, label))

# Apply tokenization to testing data
tokenized_test_data_words = []
for review_text, label in test_data:
    tokenized_review = tokenize_review_basic(review_text)
    tokenized_test_data_words.append((tokenized_review, label))

# Print the first few tokenized reviews from training data
print("First 2 Tokenized Training Reviews:")
for i in range(min(2, len(tokenized_train_data_words))):
    print(f"Review: {tokenized_train_data_words[i][0][:10]}... (first 10 words)")
    print(f"Label: {tokenized_train_data_words[i][1]}")
    print("\n")

# Print the first few tokenized reviews from testing data
print("First 2 Tokenized Testing Reviews:")
for i in range(min(2, len(tokenized_test_data_words))):
    print(f"Review: {tokenized_test_data_words[i][0][:10]}... (first 10 words)")
    print(f"Label: {tokenized_test_data_words[i][1]}")
    print("\n")

## Analyze Raw Review Lengths

In [ ]:
# Extract and calculate lengths for training data
raw_train_review_lengths = [len(review_tokens) for review_tokens, _ in tokenized_train_data_words]

# Calculate and print statistics for training data
print("Training Data Raw Review Length Statistics:")
print(f"Min length: {np.min(raw_train_review_lengths)}")
print(f"Max length: {np.max(raw_train_review_lengths)}")
print(f"Average length: {np.mean(raw_train_review_lengths):.2f}")
print(f"Median length: {np.median(raw_train_review_lengths)}")

# Extract and calculate lengths for testing data
raw_test_review_lengths = [len(review_tokens) for review_tokens, _ in tokenized_test_data_words]

# Calculate and print statistics for testing data
print("\nTesting Data Raw Review Length Statistics:")
print(f"Min length: {np.min(raw_test_review_lengths)}")
print(f"Max length: {np.max(raw_test_review_lengths)}")
print(f"Average length: {np.mean(raw_test_review_lengths):.2f}")
print(f"Median length: {np.median(raw_test_review_lengths)}")

## Analyze Label Balance


In [ ]:

# Extract labels from tokenized_train_data_words
train_labels = [label for _, label in tokenized_train_data_words]

# Use a counter to count the occurrences of each label in the training set
train_label_counts = Counter(train_labels)

# Calculate the percentage of each label in the training set
train_total = len(train_labels)

print("Training Data Label Distribution:")
for label, count in train_label_counts.items():
    percentage = (count / train_total) * 100
    print(f"Label {label}: {count} reviews ({percentage:.2f}%)")

# Repeat for all of the list
test_labels = [label for _, label in tokenized_test_data_words]
test_label_counts = Counter(test_labels)
test_total = len(test_labels)

print("\nTesting Data Label Distribution:")
for label, count in test_label_counts.items():
    percentage = (count / test_total) * 100
    print(f"Label {label}: {count} reviews ({percentage:.2f}%)")

## Identify Most Frequent Words

In [ ]:
all_raw_words = []


for review_tokens, _ in tokenized_train_data_words:
    all_raw_words.extend(review_tokens)

raw_word_counts = Counter(all_raw_words)
print("--- Top 20 Most Frequent Words (excluding special tokens and 'br') ---")

# Filter out special tokens and 'br' from word_counts
filtered_word_counts = {word: count for word, count in raw_word_counts.items()}

# Sort and get the top 20
top_20_words_cleaned = sorted(filtered_word_counts.items(), key=lambda item: item[1], reverse=True)[:20]

for word, count in top_20_words_cleaned:
    print(f"'{word}': {count}")

### Data Cleaning and Tokenization Pipeline Explanation


1.  **Data Loading Process:**
    *   Unzip and load thr data
    *   Combine the pos and neg examples
    *   Shuffle and test and training data

2.  **Basic Tokenization and clean:**
    *   Get a list of words from raw text
    *   Clean special tokens like br for html, t and s which are suffixes

3.  **Data Exploration - Review Lengths:**
    *   Training Data: Min length of 10 words, max of 2,525 words, average length of approximately 242.27 words, and median of 181 words.
    *   **Testing Data:** Min length of 6 words, max of 2,389 words, average length of approximately 236.80 words, and median of 178 words.
    *   This may indicate a large variance in review lengths, min is 6 and avf is 236

4.  **Labels count:**
    *   12,500 reviews for positive and same for negitive.

5.  **Most Frequent Words:**
    *   The top 20 most frequent words from the *basic tokenized training data* were identified. These included common English words like 'the', 'and', 'a', 'of', 'to', 'is', along with specific terms like 'movie' and 'film'.
    *   An important note, 'br', appeared frequently. This indicates the presence of HTML br tags in the original movie review texts, we cleaned it with other special tokens.


*****************************

### b) Main (50 pt)

Define 2 models, as requested. Train and eval them.

- Plot the gated model's accuracy and loss (both on train and test sets) as a function of epochs.
- Plot a confusion matrix for the model outputs on the train and test sets.
- Did you observe differences in models training process?


Note, you are allowed to use `nn.RNN` and `nn.LSTM` in your model.


In [ ]:
class VocabEmbedding(nn.Module):
    """
    Handles:
      - loading vocabulary from imdb.vocab
      - mapping word <-> index
      - embedding lookup
      - providing a method to get embedding for a single word
    """
    def __init__(self, vocab_path, embed_dim=128):
        super().__init__()

        # Load raw vocab file
        with open(vocab_path, "r", encoding="utf-8") as f:
            words = [line.strip() for line in f if line.strip()]

        # Add special tokens
        self.special_tokens = ["<pad>", "<unk>", "<sos>", "<eos>"]
        vocab = self.special_tokens + words
        word_to_token_index = {w: i for i, w in enumerate(vocab)}

        self.vocab = vocab
        self.word_to_token_index = word_to_token_index
        self.vocab_size = len(vocab)

        # Embedding layer
        self.embedding = nn.Embedding(
            num_embeddings=self.vocab_size,
            embedding_dim=embed_dim,
            padding_idx=self.word_to_token_index["<pad>"],
        )

    def forward(self, x_ids):
        """
        x_ids: LongTensor of shape [B, T] with token indices
        returns: [B, T, E]
        """
        return self.embedding(x_ids)

    def word_to_id(self, word):
        return self.word_to_token_index.get(word, self.word_to_token_index["<unk>"])

    def id_to_word(self, idx):
        return self.vocab[idx]

    def get_word_embedding(self, word):
        """
        Returns the embedding vector for a single word (1D tensor of size [E]).
        """
        self.eval()
        idx = torch.tensor([self.word_to_id(word)], dtype=torch.long, device=self.embedding.weight.device)
        with torch.no_grad():
            emb = self.embedding(idx)  # [1, E]
        return emb.squeeze(0)         # [E]

## RNN

### Define IMDB dataset

In [ ]:
class IMDBDataset(Dataset):
    def __init__(self, data, vocab_embedding, max_seq_len=256):
        self.data = data
        self.vocab_embedding = vocab_embedding
        self.max_seq_len = max_seq_len

        # Store special token IDs
        self.pad_id = self.vocab_embedding.word_to_id("<pad>")
        self.unk_id = self.vocab_embedding.word_to_id("<unk>")
        self.sos_id = self.vocab_embedding.word_to_id("<sos>")
        self.eos_id = self.vocab_embedding.word_to_id("<eos>")

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        review_tokens, label = self.data[idx]

        # Convert tokens to IDs, adding SOS and EOS
        token_ids = [self.sos_id] + [self.vocab_embedding.word_to_id(token) for token in review_tokens] + [self.eos_id]

        # Pad or truncate the sequence
        if len(token_ids) < self.max_seq_len:
            padded_token_ids = token_ids + [self.pad_id] * (self.max_seq_len - len(token_ids))
        else:
            padded_token_ids = token_ids[:self.max_seq_len]

        return torch.tensor(padded_token_ids, dtype=torch.long), torch.tensor(label, dtype=torch.long)

### Create instances

In [ ]:
vocab_path = '/content/aclImdb/imdb.vocab'
embed_dim = 128

# Instantiate VocabEmbedding
vocab_embedding = VocabEmbedding(vocab_path, embed_dim)
print(f"VocabEmbedding instantiated with vocab size: {vocab_embedding.vocab_size} and embedding dim: {vocab_embedding.embedding.embedding_dim}")

# Create IMDBDataset instances
train_dataset = IMDBDataset(tokenized_train_data_words, vocab_embedding, max_seq_len=256)
test_dataset = IMDBDataset(tokenized_test_data_words, vocab_embedding, max_seq_len=256)
print(f"IMDBDataset instances created. Train dataset size: {len(train_dataset)}, Test dataset size: {len(test_dataset)}")

# Create DataLoader instances
batch_size = 64
train_dataloader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_dataloader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)
print(f"DataLoaders created with batch size: {batch_size}")

### Define train and test loops

In [ ]:
# Device configuration
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

def train_epoch(model, dataloader, optimizer, criterion, device):
    model.train() # Set the model to training mode
    total_loss = 0
    correct_predictions = 0
    total_samples = 0

    for inputs, labels in tqdm(dataloader, desc="Training"):
        inputs, labels = inputs.to(device), labels.to(device)

        # Zero the gradients
        optimizer.zero_grad()

        # Forward pass
        outputs = model(inputs)
        loss = criterion(outputs, labels)

        # Backward pass
        loss.backward()

        # Gradient clipping to prevent exploding gradients
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=GRADIENT_CLIP_MAX_NORM)

        # Optimization step
        optimizer.step()

        total_loss += loss.item() * inputs.size(0)

        # Calculate accuracy
        _, predicted = torch.max(outputs.data, 1)
        total_samples += labels.size(0)
        correct_predictions += (predicted == labels).sum().item()

    avg_loss = total_loss / total_samples
    accuracy = correct_predictions / total_samples
    return avg_loss, accuracy

def evaluate(model, dataloader, criterion, device):
    model.eval() # Set the model to evaluation mode
    total_loss = 0
    correct_predictions = 0
    total_samples = 0

    with torch.no_grad(): # Disable gradient calculation for evaluation
        for inputs, labels in tqdm(dataloader, desc="Evaluating"):
            inputs, labels = inputs.to(device), labels.to(device)

            # Forward pass
            outputs = model(inputs)
            loss = criterion(outputs, labels)

            total_loss += loss.item() * inputs.size(0)

            # Calculate accuracy
            _, predicted = torch.max(outputs.data, 1)
            total_samples += labels.size(0)
            correct_predictions += (predicted == labels).sum().item()

    avg_loss = total_loss / total_samples
    accuracy = correct_predictions / total_samples
    return avg_loss, accuracy

### RNN class

In [ ]:
class RNNModel(nn.Module):
    def __init__(self, vocab_embedding, hidden_dim, output_dim, num_layers, dropout):
        super().__init__()
        # Settings
        self.vocab_embedding = vocab_embedding
        self.embedding_dim = vocab_embedding.embedding.embedding_dim
        self.hidden_dim = hidden_dim
        self.output_dim = output_dim
        self.num_layers = num_layers
        self.bidirectional = True

        self.rnn = nn.RNN(
            input_size=self.embedding_dim,
            hidden_size=self.hidden_dim,
            num_layers=self.num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0,
            bidirectional=self.bidirectional
        )

        fc_input_dim = self.hidden_dim * 2
        self.fc = nn.Linear(fc_input_dim, self.output_dim)
        self.dropout = nn.Dropout(dropout)

    def forward(self, text_ids):
        embedded = self.vocab_embedding(text_ids)
        output, hidden = self.rnn(embedded)

        # Mean Pooling
        mean_pooled = torch.mean(output, dim=1)

        pooled_output = self.dropout(mean_pooled)
        return self.fc(pooled_output)

### Setting hyper parameters, train & eval

In [ ]:
# Hyperparameters
HIDDEN_DIM = 128
OUTPUT_DIM = 2
NUM_LAYERS = 2
DROPOUT = 0.4
LEARNING_RATE = 0.0005
NUM_EPOCHS = 10
GRADIENT_CLIP_MAX_NORM = 5.0
WEIGHT_DECAY = 1e-4

# Instantiate
model = RNNModel(vocab_embedding, HIDDEN_DIM, OUTPUT_DIM, NUM_LAYERS, DROPOUT)
model.to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)

# Metric Storage
train_losses = []
train_accuracies = []
test_losses = []
test_accuracies = []

# Early Stopping Variables
best_test_acc = 0.0
best_model_wts = copy.deepcopy(model.state_dict())


for epoch in range(NUM_EPOCHS):
    # Training
    model.train()
    total_train_loss = 0
    train_correct = 0
    train_total = 0

    for inputs, labels in tqdm(train_dataloader, desc=f"Epoch {epoch+1} Training"):
        inputs, labels = inputs.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()

        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=GRADIENT_CLIP_MAX_NORM)
        optimizer.step()

        total_train_loss += loss.item() * inputs.size(0)
        _, predicted = torch.max(outputs.data, 1)
        train_total += labels.size(0)
        train_correct += (predicted == labels).sum().item()

    avg_train_loss = total_train_loss / train_total
    train_acc = train_correct / train_total
    train_losses.append(avg_train_loss)
    train_accuracies.append(train_acc)

    #  Evaluation
    model.eval()
    total_test_loss = 0
    test_correct = 0
    test_total = 0

    with torch.no_grad():
        for inputs, labels in test_dataloader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            loss = criterion(outputs, labels)

            total_test_loss += loss.item() * inputs.size(0)
            _, predicted = torch.max(outputs.data, 1)
            test_total += labels.size(0)
            test_correct += (predicted == labels).sum().item()

    avg_test_loss = total_test_loss / test_total
    test_acc = test_correct / test_total
    test_losses.append(avg_test_loss)
    test_accuracies.append(test_acc)

    print(f"Epoch {epoch+1}/{NUM_EPOCHS} | Train Loss: {avg_train_loss:.4f}, Train Acc: {train_acc:.4f} | Test Loss: {avg_test_loss:.4f}, Test Acc: {test_acc:.4f}")

    #  Save Best Model
    if test_acc > best_test_acc:
        best_test_acc = test_acc
        best_model_wts = copy.deepcopy(model.state_dict())
        print(f"  --> New Best Model Saved! (Acc: {best_test_acc:.4f})")

#  Load Best Weights
print(f"\nLoading best model weights with Test Accuracy: {best_test_acc:.4f}")
model.load_state_dict(best_model_wts)
print("Training complete.")

# Save the best model
torch.save(best_model_wts, 'HW2_000000000_000000000_RNN.pkl')
print("Best model weights saved to HW2_000000000_000000000__RNN.pkl")


### Plot accuracy and loss

In [ ]:
# Create a figure with two subplots
plt.figure(figsize=(12, 5))

# Subplot 1: Loss
plt.subplot(1, 2, 1) # 1 row, 2 columns, first plot
plt.plot(range(1, NUM_EPOCHS + 1), train_losses, label='Train Loss')
plt.plot(range(1, NUM_EPOCHS + 1), test_losses, label='Test Loss')
plt.title('Training and Testing Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.grid(True)

# Subplot 2: Accuracy
plt.subplot(1, 2, 2) # 1 row, 2 columns, second plot
plt.plot(range(1, NUM_EPOCHS + 1), train_accuracies, label='Train Accuracy')
plt.plot(range(1, NUM_EPOCHS + 1), test_accuracies, label='Test Accuracy')
plt.title('Training and Testing Accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()
plt.grid(True)

plt.tight_layout() # Adjust layout to prevent overlapping titles/labels
plt.show()


### Confusion Matrix - Training Data

In [ ]:

# Get predictions and true labels for the training set
model.eval() # Set model to evaluation mode
train_preds = []
train_true_labels = []

with torch.no_grad():
    for inputs, labels in tqdm(train_dataloader, desc="Collecting Train Predictions"):
        inputs, labels = inputs.to(device), labels.to(device)
        outputs = model(inputs)
        _, predicted = torch.max(outputs.data, 1)
        train_preds.extend(predicted.cpu().numpy())
        train_true_labels.extend(labels.cpu().numpy())

# Calculate confusion matrix for training data
cm_train = confusion_matrix(train_true_labels, train_preds)

# Plot confusion matrix for training data
plt.figure(figsize=(6, 5))
display_cm_train = ConfusionMatrixDisplay(confusion_matrix=cm_train, display_labels=[0, 1])
display_cm_train.plot(cmap=plt.cm.Blues)
plt.title('Confusion Matrix - Training Data (RNN)')
plt.show()


### Confusion Matrix - Testing Data

In [ ]:
test_preds = []
test_true_labels = []

with torch.no_grad():
    for inputs, labels in tqdm(test_dataloader, desc="Collecting Test Predictions"):
        inputs, labels = inputs.to(device), labels.to(device)
        outputs = model(inputs)
        _, predicted = torch.max(outputs.data, 1)
        test_preds.extend(predicted.cpu().numpy())
        test_true_labels.extend(labels.cpu().numpy())

# Calculate confusion matrix for test data
cm_test = confusion_matrix(test_true_labels, test_preds)

# Plot confusion matrix for test data
plt.figure(figsize=(6, 5))
display_cm_test = ConfusionMatrixDisplay(confusion_matrix=cm_test, display_labels=[0, 1])
display_cm_test.plot(cmap=plt.cm.Blues)
plt.title('Confusion Matrix - Testing Data (RNN)')
plt.show()

*****************************

*****************************

## LSTM

In [ ]:
class IMDBDataset(Dataset):
    def __init__(self, folder_path, vocab, max_len=200):
        self.vocab = vocab
        self.max_len = max_len
        self.samples = []

        for label_name in ['pos', 'neg']:
            label_val = 1.0 if label_name == 'pos' else 0.0
            dir_path = os.path.join(folder_path, label_name)

            if os.path.exists(dir_path):
                for filename in os.listdir(dir_path):
                    if filename.endswith(".txt"):
                        file_path = os.path.join(dir_path, filename)
                        with open(file_path, 'r', encoding='utf-8') as f:
                            text = f.read()
                            self.samples.append((text, label_val))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        raw_text, label = self.samples[idx]

        words = raw_text.lower().split()

        token_ids = [self.vocab.word_to_id(w) for w in words]

        # Padding / Truncating
        current_len = len(token_ids)
        if current_len < self.max_len:
            padding = [self.vocab.word_to_id("<pad>")] * (self.max_len - current_len)
            token_ids = token_ids + padding
        else:
            token_ids = token_ids[:self.max_len]

        return torch.tensor(token_ids, dtype=torch.long), torch.tensor(label, dtype=torch.float)


os.makedirs("dummy_data/pos", exist_ok=True)
os.makedirs("dummy_data/neg", exist_ok=True)

with open("dummy_vocab.txt", "w") as f:
    f.write("movie\nis\ngood\nbad\nexcellent\nterrible")

with open("dummy_data/pos/review1.txt", "w") as f:
    f.write("The movie is excellent")

with open("dummy_data/neg/review2.txt", "w") as f:
    f.write("The movie is terrible")

vocab_obj = VocabEmbedding("dummy_vocab.txt")


train_ds = IMDBDataset("dummy_data", vocab_obj, max_len=10) # אורך קצר לבדיקה


x, y = train_ds[0]

print(f"Original Text (approx): 'The movie is excellent'")
print(f"Tensor Shape: {x.shape}")
print(f"Tensor Content: {x}")
print(f"Label: {y}")

import shutil
shutil.rmtree("dummy_data")
os.remove("dummy_vocab.txt")

In [ ]:
class SentimentLSTM(nn.Module):
    def __init__(self, vocab_embedding, hidden_dim, output_dim, n_layers, bidirectional, dropout):
        super().__init__()

        self.embedding = vocab_embedding
        embedding_dim = vocab_embedding.embedding.embedding_dim


        self.lstm = nn.LSTM(embedding_dim,
                            hidden_dim,
                            num_layers=n_layers,
                            bidirectional=bidirectional,
                            dropout=dropout,
                            batch_first=True)

        scale = 2 if bidirectional else 1
        self.fc = nn.Linear(hidden_dim * scale, output_dim)

        self.dropout = nn.Dropout(dropout)

    def forward(self, text):
        # text: [batch size, sent len]

        # המרה לווקטורים
        embedded = self.embedding(text)


        output, (hidden, cell) = self.lstm(embedded)

        if self.lstm.bidirectional:
            hidden = self.dropout(torch.cat((hidden[-2,:,:], hidden[-1,:,:]), dim = 1))
        else:
            hidden = self.dropout(hidden[-1,:,:])

        return self.fc(hidden)

In [ ]:
def train(model, iterator, optimizer, criterion):
    epoch_loss = 0
    epoch_acc = 0
    model.train()

    for batch_idx, (text, labels) in enumerate(iterator):

        device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        text, labels = text.to(device), labels.to(device)

        optimizer.zero_grad() # איפוס נגזרות

        predictions = model(text).squeeze(1) # הרצת המודל
        loss = criterion(predictions, labels) # חישוב השגיאה

        # חישוב דיוק (Accuracy)
        rounded_preds = torch.round(torch.sigmoid(predictions))
        correct = (rounded_preds == labels).float()
        acc = correct.sum() / len(correct)

        loss.backward() # חישוב נגזרות (Backprop)
        optimizer.step() # עדכון משקולות

        epoch_loss += loss.item()
        epoch_acc += acc.item()

    return epoch_loss / len(iterator), epoch_acc / len(iterator)

def evaluate(model, iterator, criterion):
    epoch_loss = 0
    epoch_acc = 0
    model.eval() # מעביר למצב הערכה

    with torch.no_grad(): # לא מחשבים נגזרות (חוסך זיכרון)
        for text, labels in iterator:
            device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
            text, labels = text.to(device), labels.to(device)

            predictions = model(text).squeeze(1)
            loss = criterion(predictions, labels)

            rounded_preds = torch.round(torch.sigmoid(predictions))
            correct = (rounded_preds == labels).float()
            acc = correct.sum() / len(correct)

            epoch_loss += loss.item()
            epoch_acc += acc.item()

    return epoch_loss / len(iterator), epoch_acc / len(iterator)

In [ ]:
# Hyperparameters
BATCH_SIZE = 64
HIDDEN_DIM = 256
OUTPUT_DIM = 1
N_LAYERS = 2
BIDIRECTIONAL = True
DROPOUT = 0.5
LR = 0.001
EPOCHS = 5

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
# Path definitions
base_path = '/content/aclImdb'
vocab_path = os.path.join(base_path, 'imdb.vocab')

# Vocab initialization
try:
    vocab_obj = VocabEmbedding(vocab_path)
    print("Vocab loaded successfully!")
except FileNotFoundError:
    print(f"Warning: could not find vocab at {vocab_path}. Please check where 'imdb.vocab' is.")

# Data loading
train_data = IMDBDataset(os.path.join(base_path, 'train'), vocab_obj)
test_data = IMDBDataset(os.path.join(base_path, 'test'), vocab_obj)

print(f"Train samples: {len(train_data)}")
print(f"Test samples: {len(test_data)}")

# Global lists for plotting
train_accs = []
val_accs = []
train_losses = []
val_losses = []

# DataLoaders and Model
if len(train_data) > 0:
    train_iterator = DataLoader(train_data, batch_size=BATCH_SIZE, shuffle=True)
    test_iterator = DataLoader(test_data, batch_size=BATCH_SIZE)

    model = SentimentLSTM(vocab_obj, HIDDEN_DIM, OUTPUT_DIM, N_LAYERS, BIDIRECTIONAL, DROPOUT)
    model = model.to(device)

    optimizer = optim.Adam(model.parameters(), lr=LR)
    criterion = nn.BCEWithLogitsLoss().to(device)

    print(f"Starting training on {device}...")
    for epoch in range(EPOCHS):
        train_loss, train_acc = train(model, train_iterator, optimizer, criterion)
        valid_loss, valid_acc = evaluate(model, test_iterator, criterion)

        # Store metrics
        train_accs.append(train_acc * 100)
        val_accs.append(valid_acc * 100)
        train_losses.append(train_loss)
        val_losses.append(valid_loss)

        print(f'Epoch: {epoch+1:02} | Train Loss: {train_loss:.3f} | Train Acc: {train_acc*100:.2f}% | Val. Loss: {valid_loss:.3f} | Val. Acc: {valid_acc*100:.2f}%')

In [ ]:
torch.save(model.state_dict(), 'HW2_000000000_000000000_LSTM.pt')
print("Model weights saved successfully!")

In [ ]:
# train_accs, val_accs, train_losses, val_losses are populated in the training cell

epochs = range(1, len(train_accs) + 1)

# Create plots
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))

# Accuracy Plot
ax1.plot(epochs, train_accs, 'bo-', label='Train Acc')
ax1.plot(epochs, val_accs, 'ro-', label='Validation Acc')
ax1.set_title('Accuracy over Epochs')
ax1.set_xlabel('Epochs')
ax1.set_ylabel('Accuracy (%)')
ax1.legend()
ax1.grid(True)

# Loss Plot
ax2.plot(epochs, train_losses, 'bo-', label='Train Loss')
ax2.plot(epochs, val_losses, 'ro-', label='Validation Loss')
ax2.set_title('Loss over Epochs')
ax2.set_xlabel('Epochs')
ax2.set_ylabel('Loss')
ax2.legend()
ax2.grid(True)

plt.show()

In [ ]:
def plot_confusion_matrix(model, iterator):
    model.eval()
    all_preds = []
    all_labels = []

    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

    with torch.no_grad():
        for text, labels in iterator:
            text, labels = text.to(device), labels.to(device)
            predictions = model(text).squeeze(1)
            rounded_preds = torch.round(torch.sigmoid(predictions))

            all_preds.extend(rounded_preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    # matrix calculation
    cm = confusion_matrix(all_labels, all_preds)

    # draw
    plt.figure(figsize=(6, 5))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=['Neg', 'Pos'], yticklabels=['Neg', 'Pos'])
    plt.ylabel('Actual')
    plt.xlabel('Predicted')
    plt.title('Confusion Matrix')
    plt.show()

    # report
    print("Classification Report:")
    print(classification_report(all_labels, all_preds, target_names=['Negative', 'Positive']))

plot_confusion_matrix(model, test_iterator)

## c) Discussion (10 pt)

LSTM:


Architecture: We selected a Bidirectional LSTM to capture long-term dependencies and context from both directions (e.g., handling negations effectively), overcoming the vanishing gradient problem of standard RNNs.

Hyperparameters: We used a hidden dimension of 256 with 2 layers to provide sufficient model capacity, and applied Dropout (0.5) to act as regularization and prevent overfitting.

Results: The model was optimized using Adam and achieved a validation accuracy of ~82%, significantly surpassing the 70% threshold. The training logs show stable convergence with no significant signs of overfitting.


RNN:
We selected a Bidirectional RNN the same reason of LSTM. we used 2 layers bc it gets really hard training a deep RNN.

Hyperparameters: We tried a different combinations of values. we did learning rate of 0.0005, we read in the internet and they say 0.002 so we adjusted. same for dropout for stopping overtfitting, also we added weight decay to penalizes large weights and prevent overfitting too. we also did gradient clip to prevent exploding gradients.

Results: The model had train accuracy of 90% and test accuracy of 85%


# Question 3 - Dry (10pt)
Answer the following questions, explain your answers.

1. Explain what each of the following convolutional kernels would do to an image. Match each one to a transformation below.

- Kernel 1: $\begin{pmatrix} 0 & 0 & 0 \\ 0 & 1 & 0 \\ 0 & 0 & 0\end{pmatrix}$

- Kernel 2: $\begin{pmatrix} 0 & 2.5 & 0 \\ 0 & 0 & 0 \\ 0 & -2.5 & 0\end{pmatrix}$

- Kernel 3: $\begin{pmatrix} 0 & -1 & 0 \\ -1 & 5 & -1 \\ 0 & -1 & 0\end{pmatrix}$

2. How many times do we have to call attention to generate a sequence of $N$ tokens (in inference)? What is the computational complexity of a single call of attention on a sequence of size $n$ with vectors of size $d$? Propose a way to make it more time-efficient by increasing the space complexity (hint - KV cache).

3. Explain the difference between a transformer encoder and decoder. Which would be more useful for solving question 2?

# Q3 a.

**Kernel 1:** $\begin{pmatrix} 0 & 0 & 0 \\ 0 & 1 & 0 \\ 0 & 0 & 0\end{pmatrix}$

*   **Explanation:** This kernel places a weight of 1 on the central pixel and 0 on all surrounding pixels. When convolved with an image, each output pixel will simply be the value of the corresponding input pixel. It effectively copies the image without alteration. and we see in the left pic
*   **Effect:** Identity

**Kernel 2:** $\begin{pmatrix} 0 & 2.5 & 0 \\ 0 & 0 & 0 \\ 0 & -2.5 & 0\end{pmatrix}$

*   **Explanation:** This kernel has a positive weight above the center and a negative weight below it. It will highlight differences in pixel intensity in the vertical direction. So basically, it will emphasize transitions from dark to light or vice versa along vertical lines, indicating vertical edges. The large magnitude of 2.5 amplifies these differences. as we see in the middle pic
*   **Effect:** Edge Detection

**Kernel 3:** $\begin{pmatrix} 0 & -1 & 0 \\ -1 & 5 & -1 \\ 0 & -1 & 0\end{pmatrix}$

*   **Explanation:** This kernel has a large positive weight at the center and negative weights on its immediate neighbors . This configuration has  increases the contrast between the central pixel and its surroundings. By emphasizing the difference, it makes details more noticeable.
*   **Effect:** Sharpening

# Q3 b.

1.  **Number of Attention Calls for N Tokens in Inference:**
    In a transformer decoder during inference, each new token is generated sequentially. To generate the first token, attention is called once. To generate the second token, attention is called again, and so forth. so for sequence in length N we need to call attention N times.

2.  **Computational Complexity of a Single Attention Call:**
    For a single attention call on a sequence of size n with vectors of size d, the self-attention mechanism involves several matrix multiplications:

    *   Query (Q), Key (K), Value (V)
    *   Calculating Q@Kᵀ: Q is n x d and Kᵀ is d x n. Their product is n x n. This matrix multiplication has a complexity of O(n * d * n) which is O(n² * d).
    *   Dividing by sqrt(d) is O(n²).
    *   Applying softmax to the n x n matrix is O(n²).
    *   Caclulating attention @ V: The n x n attention weights matrix is multiplied by the n x d Value matrix. This results in an n x d output matrix. The complexity is O(n * n * d) which is O(n² * d).

    In total we got **O(n² * d)**

3.  **Time-Efficient solution:**

    In the decoding part, when generating the n-th token, the model needs to attend to all n-1 previously generated tokens. So we can use a cache (called KV cache) to store the previous n-1 previously generated tokens.
    So, when generating a new token we compute the K and V for the current token, we concat them to the previous tokens in the cache and use it with Q to calculate the attention

    **Time Complexity:**

    Instead of recalculating K and V for the entire sequence at each step, we only compute them once for each token and store them. So the complexity for computing the attention output for one token given n previous tokens becomes **O(n * d)**.

    **Increased space complexity:** The trade off is that the K and V matrices for all previously generated tokens need to be stored in memory. This increases the space complexity from O(d) per token to O(n * d) for n tokens.

# Q3.c - transformer encoder vs. decoder

*   **Transformer Encoder:**
    *  The encoder's role is to process an input sequence and produce a rich, contextualized representation of each token in that sequence. It focuses on understanding the entire input. It's often used for tasks where a fixed length representation of the input is needed, such as classification or feature extraction. like BERT
    *   The encoder uses self attention .This allows every token in the input sequence to attend to every other token in the same input sequence. There is no masking applied, meaning each position can see all past and future positions in the input when computing its representation. This is crucial for understanding the full meaning of the input.
    *   Each encoder layer typically consists of two sub layers: a Multi head self attention layer and a position wise feed forward network. Each of these sub layers also employs a residual connection followed by layer normalization.

*   **Transformer Decoder:**
    *   The decoder's role is to generate an output sequence one token at a time, based on the contextualized input representation provided by the encoder and the tokens it has generated so far. It's designed for sequence generation tasks like summarization, translation. like GPT
    *   The decoder has three sub-layers:
        1.  The main difference in the decoder is masked multi head self attention. This is similar to the encoder's self attention but it's masked. When computing the representation for a given output token, it can only attend to previous output tokens and the token itself. It cannot look at future tokens in the output sequence. This ensures that the generation process is auto regressive meaning, genegrate only based on the past.
        2.  Another thing is multi head cross attention. This layer allows the decoder to attend to the output of the encoder. This helps the decoder align its output generation with the relevant parts of the input sequence. The query comes from the masked self attention output, while the keys and values come from the encoder's output.
        3.  Similar to the encoder, position wise feed forward Network processes the output of the cross attention layer.

**So which one would we prefer for Q2?**

For a sentiment analysis task like Q2, which involves classifying an entire input review as positive or negative, the encoder would be more useful. encoder is better when talking about getting a context from a full sentence, decoder knows only the past and cant look at the entire sentence. also, encoder is the way when we need a classifying job. Decoder would be great for generating because of the masked multihead attention.